In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from src.features import add_features

food = pd.read_csv('../data/processed/food_clean.csv', parse_dates=['month'])
fx = pd.read_csv('../data/processed/fx_monthly.csv', parse_dates=['month'])
pms = pd.read_csv('../data/processed/pms_monthly.csv', parse_dates=['month'])
# May 2023 is not in the reports we have (it only appears in the May 2024 report).
# Value from NBS (national average, NGN per litre): 238.11
if pd.Timestamp('2023-05-01') not in set(pms['month']):
    pms = pd.concat([pms, pd.DataFrame({'month': [pd.Timestamp('2023-05-01')], 'pms_price': [238.11]})])
    pms = pms.sort_values('month').reset_index(drop=True)

df = food.merge(fx, on='month', how='left').merge(pms, on='month', how='left')
df = df[(df['month'] >= '2023-01-01') & (df['month'] <= pms['month'].max())]   # window: Jan 2023 to last PMS month

print(df.isna().sum())       # usd_ngn and pms_price must show 0 missing
print(df['month'].min(), df['month'].max(), df.shape)

master = add_features(df)
master.to_csv('../data/processed/master.csv', index=False)
print(master.shape)
print(master.head())